# Data Exploration and Processing
## Shopify

### By: Makaled Consulting Group

In [ ]:
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_csv("shopify_sales_dataset_ml_eda.csv")

df.head()

Initial Data Dictionary (to understand the working dataset before applying modifications)

| Field | Data Type | Length | Key | Nullable | Description |
| ----- | --------- | ------ | --- | -------- | ----------- |
| `order_id` | INT | 5 | PK | NO | Order identifier |
| `order_date` | DATE | 10 | | NO | Order date |
| `customer_id` | INT | 5 | FK | NO | Customer identifier |
| `product_id` | INT | 4 | FK | NO | Product identifier |
| `product_category` | VARCHAR | 11 | | NO | Product category |
| `product_price` | DECIMAL(5,2) | 5 | | NO | Product list price |
| `discount_percent` | INT | 2 | | NO | Applied discount percentage |
| `quantity` | INT | 1 | | NO | Quantity of units purchased |
| `customer_country` | VARCHAR | 9 | | NO | Customer country |
| `traffic_source` | VARCHAR | 12 | | NO | Acquisition traffic channel |
| `payment_method` | VARCHAR | 16 | | NO | Payment method |
| `shipping_cost` | DECIMAL(4,2) | 4 | | NO | Shipping cost |
| `rating` | DECIMAL(2,1) | 2 | | NO | Customer rating |
| `is_returned` | TINYINT | 1 | | NO | Indicates if the order was returned (1 = yes, 0 = no) |
| `discounted_price` | DECIMAL(5,2) | 5 | | NO | Unit price after discount |
| `revenue` | DECIMAL(6,2) | 6 | | NO | Order revenue (discounted price x quantity) |
| `profit` | DECIMAL(6,2) | 6 | | NO | Order profit (revenue - shipping cost) |
| `return_date` | DATE | 10 | | YES | Return date |
| `return_reason` | VARCHAR | 23 | | YES | Return reason |
| `refund_status` | VARCHAR | 12 | | YES | Refund status |
| `days_to_return` | INT | 2 | | YES | Days between order and return |


## Original Dataset: Shopify Sales

Dataset Size Inspection

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Data Types Mapping

In [ ]:
df.info()

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"]) # Convert order_date data type to datetime

Missing Values Detection

In [ ]:
df.isnull().sum()

Duplicates Detection

In [ ]:
df.duplicated().sum()

Business Logic Analysis

In [ ]:
df.describe()

Recalculate Revenue

In [ ]:
df.drop(columns=["revenue"], inplace=True)

In [ ]:
df['revenue'] = (df['product_price'] * (1 - df['discount_percent'] / 100) * df['quantity'])

Recalculate Profit

In [ ]:
df.drop(columns=["profit"], inplace=True)

In [ ]:
# Estimated base cost (e.g., 50% of revenue)
estimated_cost = df['revenue'] * 0.50

# If the product was returned, add an additional 30% revenue penalty in costs
return_penalty = np.where(df['is_returned'] == True, df['revenue'] * 0.30, 0)

# Final profit
df['profit'] = df['revenue'] - estimated_cost - df['shipping_cost'] - return_penalty

## Returns Detail Dataset

**A merge will be applied with this table**

A table with return details was generated. This complements the information within the project scope.

In [ ]:
df_returns = pd.read_csv("shopify_returns_clean.csv")
df_returns.head()

Exploration of Missing Values in New Table

In [ ]:
df_returns.isnull().sum()

Dataset Size Inspection

In [ ]:
print("Rows:", df_returns.shape[0])
print("Columns:", df_returns.shape[1])

Duplicates Detection

In [ ]:
df_returns.duplicated().sum()

### Joining Main and Complementary Tables via Merge

In [ ]:
df_final = pd.merge(df, df_returns, on='order_id', how='left')

df_final.head()

### Feature Engineering
Month and Day of the Week

In [ ]:
df_final['order_month'] = df_final['order_date'].dt.strftime('%Y-%m')
df_final['order_day']   = df_final['order_date'].dt.day_name()        # 'Monday', 'Tuesday'...
df_final['weekday_num'] = df_final['order_date'].dt.dayofweek         # 0 = Monday ... 6 = Sunday
df_final['is_weekend']  = df_final['weekday_num'] >= 5

df_final.head()

In [ ]:
df_final.info()

Days Elapsed Between Order and Return

In [ ]:
df_final["return_date"] = pd.to_datetime(df_final["return_date"]) # Convert return_date data type

In [ ]:
df_final['return_days'] = (df_final['return_date'] - df_final['order_date']).dt.days

df_final.head()

Average Order Value / Average Ticket (Isolated Metric)

In [ ]:
mean_revenue = df_final['revenue'].mean()
print(f"Average Order Value: {mean_revenue:.2f}")

Column Removal

In [ ]:
# Traffic Source is not required for the project context
df_final = df_final.drop("traffic_source", axis=1)

Data Type Modification: is_returned

In [ ]:
df_final['is_returned'] = df_final['is_returned'].astype('boolean')
df_final.head()

Exploration of Changes

In [ ]:
df_final.info()

Missing Values Identification in Final DataFrame

In [ ]:
df_final.isnull().sum()

In [ ]:
# Return reason is filled with "NA" because if no return occurred, it does not apply.
df_final['return_reason'] = df_final['return_reason'].fillna('NA')

# Refund Status is filled with "NA" because if no return occurred, no refund exists.
df_final['refund_status'] = df_final['refund_status'].fillna('NA')

return_date and return_days are not modified, as it is not possible to insert a date or calculate return days for a non-existent return.

In [ ]:
df_final.isnull().sum()

In [ ]:
df_final.info()

The dataframe now meets the requirements for data types, cleaning, and handling of applicable missing values.

### Correlation Matrix

In [ ]:
# Column correlation
cols_corr = [
    "product_price", "discount_percent", "quantity", "shipping_cost",
    "rating", "is_returned", "discounted_price", "revenue", "profit"
]

df_num = df_final[cols_corr].copy()
df_num["is_returned"] = df_num["is_returned"].astype(int)

corr = df_num.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".4f", cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()